# Continuous profiling with OpenTelemetry: the fourth signalRun the cells in order. They install the OpenTelemetry Profiling integration through the FleetAPI, start an Elastic Agent and Spring PetClinic in Docker, generate traffic, query the profilesamples with ES|QL, and delete everything again at the end.Before you start: Elasticsearch and Kibana 9.4 or later, Docker running on a Linux host orDocker Desktop, and a `.env` copied from `.env.example`. Create the API key as a superuser so itcan initialize Universal Profiling storage and use Fleet.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import osimport subprocessimport timefrom datetime import datetime, timedelta, timezoneimport requestsfrom dotenv import load_dotenvfrom elasticsearch import Elasticsearchload_dotenv()ES_URL = os.environ["ELASTICSEARCH_URL"]API_KEY = os.environ["ELASTICSEARCH_API_KEY"]KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")SPACE = os.getenv("KIBANA_SPACE", "default")BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"AGENT_VERSION = os.getenv("AGENT_VERSION", "9.5.3")TRAFFIC_MINUTES = int(os.getenv("TRAFFIC_MINUTES", "4"))HEADERS = {"Authorization": f"ApiKey {API_KEY}", "kbn-xsrf": "true", "Content-Type": "application/json"}PACKAGE = "profiling_otel"POLICY_NAME = "OTel Profiling Preview Lab"SAMPLES_PER_SECOND = 20es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=60)def kbn(method, path, payload=None):    response = requests.request(method, f"{BASE}{path}", headers=HEADERS, json=payload, timeout=120)    if response.status_code not in (200, 201, 202):        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:500]}")    return response.json() if response.text else {}def ts(moment):    return moment.isoformat(timespec="seconds").replace("+00:00", "Z")print(f"elasticsearch {es.info()['version']['number']}")

## 2. Initialize Universal Profiling storageSame as **Set up Universal Profiling** in Kibana. Does nothing if storage already exists.

In [ ]:
status = kbn("GET", "/api/profiling/setup/es_resources")if not status.get("has_setup"):    kbn("POST", "/api/profiling/setup/es_resources", {})    for _ in range(30):        time.sleep(5)        if kbn("GET", "/api/profiling/setup/es_resources").get("has_setup"):            breakprint("has_setup:", kbn("GET", "/api/profiling/setup/es_resources").get("has_setup"))

## 3. Install the integration`prerelease=true` makes the technical preview package visible.

In [ ]:
package = kbn("GET", f"/api/fleet/epm/packages/{PACKAGE}?prerelease=true")["item"]PACKAGE_VERSION = package["version"]if package["status"] != "installed":    kbn("POST", f"/api/fleet/epm/packages/{PACKAGE}/{PACKAGE_VERSION}", {"force": True})print(f"{package['title']} {PACKAGE_VERSION} installed")

## 4. Create the profiling policySame as **Add OpenTelemetry Profiling** in Kibana: a new Agent policy with one receiver and`samples_per_second` set to 20.

In [ ]:
existing = kbn("GET", f"/api/fleet/agent_policies?kuery=name:\"{POLICY_NAME}\"")["items"]agent_policy = existing[0] if existing else kbn(    "POST", "/api/fleet/agent_policies",    {"name": POLICY_NAME, "namespace": "default", "monitoring_enabled": []},)["item"]AGENT_POLICY_ID = agent_policy["id"]template = package["policy_templates"][0]variables = {v["name"]: v["default"] for v in template.get("vars", []) if v.get("default") is not None}variables["samples_per_second"] = SAMPLES_PER_SECONDinputs = {    f"{template['name']}-{template['input']}": {        "enabled": True,        "streams": {f"{PACKAGE}.{template['name']}": {"enabled": True, "vars": variables}},    }}package_policies = kbn(    "GET", f"/api/fleet/package_policies?kuery=ingest-package-policies.policy_id:\"{AGENT_POLICY_ID}\"")["items"]package_policy = package_policies[0] if package_policies else kbn(    "POST", "/api/fleet/package_policies",    {"name": "otel-profiling-preview-lab", "policy_id": AGENT_POLICY_ID,     "package": {"name": PACKAGE, "version": PACKAGE_VERSION}, "inputs": inputs},)["item"]PACKAGE_POLICY_ID = package_policy["id"]print(f"agent policy {AGENT_POLICY_ID} with {package_policy['name']}")

## 5. Get the enrollment valuesThe Fleet Server URL and the policy's enrollment token go into `.env.docker` for the Agentcontainer.

In [ ]:
keys = [k for k in kbn("GET", f"/api/fleet/enrollment_api_keys?kuery=policy_id:\"{AGENT_POLICY_ID}\"")["items"] if k.get("active")]if not keys:    keys = [kbn("POST", "/api/fleet/enrollment_api_keys", {"policy_id": AGENT_POLICY_ID})["item"]]ENROLLMENT_TOKEN = kbn("GET", f"/api/fleet/enrollment_api_keys/{keys[0]['id']}")["item"]["api_key"]hosts = kbn("GET", "/api/fleet/fleet_server_hosts")["items"]FLEET_URL = next((h for h in hosts if h.get("is_default")), hosts[0])["host_urls"][0]with open(".env.docker", "w") as handle:    handle.write(f"FLEET_URL={FLEET_URL}\nFLEET_ENROLLMENT_TOKEN={ENROLLMENT_TOKEN}\nAGENT_VERSION={AGENT_VERSION}\n")print(f"FLEET_URL={FLEET_URL}")

## 6. Start the Agent and PetClinic`docker-compose.yml` runs the Agent with the permissions the eBPF profiler needs and PetClinic asa plain jar on a HotSpot JVM. The first run builds the PetClinic image, which takes a few minutes.

In [ ]:
COMPOSE = ["docker", "compose", "--env-file", ".env.docker"]def compose(*args):    result = subprocess.run(COMPOSE + list(args), capture_output=True, text=True)    print(result.stdout, result.stderr)    result.check_returncode()compose("up", "-d", "--build")for _ in range(60):    try:        if requests.get("http://localhost:8080/", timeout=3).status_code == 200:            print("petclinic is up on http://localhost:8080")            break    except requests.RequestException:        pass    time.sleep(5)else:    raise RuntimeError("petclinic did not answer on :8080; check `docker logs petclinic`")

## 7. Wait for the AgentTakes about a minute. The Agent shows up as Healthy in **Fleet > Agents**.

In [ ]:
AGENT_ID = Nonefor _ in range(36):    agents = kbn("GET", f"/api/fleet/agents?kuery=policy_id:\"{AGENT_POLICY_ID}\"")["items"]    if agents and agents[0]["status"] == "online":        AGENT_ID = agents[0]["id"]        break    time.sleep(10)if AGENT_ID is None:    raise RuntimeError("agent is not online yet; rerun this cell or check `docker logs elastic-agent`")print(f"agent {AGENT_ID} online")

## 8. Generate trafficFour minutes of owner searches, owner pages, and the vets list.

In [ ]:
PATHS = ["/owners?lastName=", "/vets.html", "/owners/find"] + [f"/owners/{i}" for i in range(1, 11)]TRAFFIC_START = datetime.now(timezone.utc).replace(microsecond=0)deadline = TRAFFIC_START + timedelta(minutes=TRAFFIC_MINUTES)session = requests.Session()count = 0while datetime.now(timezone.utc) < deadline:    for path in PATHS:        session.get(f"http://localhost:8080{path}", timeout=10)        count += 1TRAFFIC_END = datetime.now(timezone.utc).replace(microsecond=0)print(f"{count:,} requests between {ts(TRAFFIC_START)} and {ts(TRAFFIC_END)}")

## 9. Query the samples with ES|QLThe profiler ships samples in batches, so the cell waits a minute first. This is the article'squery: `Stacktrace.count` is the number of CPU samples in each event, so the sum per minute is asample count. Open **Infrastructure > Universal Profiling > Flamegraphs** with the same time rangeand `process.executable.name : "java"` to see the stacks behind these numbers.

In [ ]:
time.sleep(60)RANGE = {"range": {"@timestamp": {"gte": ts(TRAFFIC_START), "lte": ts(TRAFFIC_END)}}}print(es.esql.query(query="""FROM profiling-events-all| WHERE process.executable.name == "java"| STATS samples = SUM(Stacktrace.count)  BY minute = BUCKET(@timestamp, 1 minute)| SORT minute""", format="txt", filter=RANGE).body)

## 10. Clean upStops the containers, unenrolls the Agent, and deletes the two policies. The `profiling-*`indices stay, since Kibana manages them.

In [ ]:
compose("down")kbn("POST", f"/api/fleet/agents/{AGENT_ID}/unenroll", {"revoke": True})kbn("POST", "/api/fleet/package_policies/delete", {"packagePolicyIds": [PACKAGE_POLICY_ID], "force": True})kbn("POST", "/api/fleet/agent_policies/delete", {"agentPolicyId": AGENT_POLICY_ID, "force": True})print("cleaned up")